# Agent Chat

## Section 0 - ECH Provisioning

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTICSEARCH_USERNAME=$(terraform -chdir=terraform output elasticsearch_username)
ELASTICSEARCH_PASSWORD=$(terraform -chdir=terraform output -raw elasticsearch_password)
ELASTICSEARCH_URL=$(terraform -chdir=terraform output -raw elasticsearch_url)
EOF
echo "Done."

## Section 1 - Agent Chat: Natural Language Query Writing (Kibana GUI)

### Section 1A - Synthetic Data Generation

In [ ]:
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
import os
from agent_chat.synthetic_data import run
    
INDEX_NAME = "demo-ecommerce"

load_dotenv(override=True)
es = Elasticsearch(
        hosts=[os.getenv("ELASTICSEARCH_URL")],
        request_timeout=180,
        basic_auth=(os.getenv("ELASTICSEARCH_USERNAME"), os.getenv("ELASTICSEARCH_PASSWORD"))
)
print(f'Client connected: {es.ping()}')

run(es, INDEX_NAME, count=5000)

### Section 1B - Elastic Agent Queries

#### Standard aggregation — top categories by revenue (STATS + SORT + LIMIT)

*"What are the top 5 product categories by total revenue?"*

![Query 1_1](assets/screenshots/query1_1.png)

#### Time-series — daily order volume trend with BUCKET

*"How has daily order volume trended over the past 3 months?"*

![Query 1_2](assets/screenshots/query1_2.png)

#### Multi-dimension grouping — regions × order status

*"Which regions have the highest average order value, broken down by order status?"*

![Query 1_3](assets/screenshots/query1_3.png)

#### Semantic search — home office products crossing category boundaries via product_description

*"Find products that someone setting up a home office would be most likely to buy.  Structure this is as a semantic search"*

![Query 1_4](assets/screenshots/query1_4.png)

#### Dashboard generation — multi-panel Kibana dashboard built directly from a natural language request

*"Build me a Kibana dashboard with panels for revenue by category, daily order trend, order status breakdown, and top regions by average order value"*

![Query 1_5](assets/screenshots/query1_5.png)
![Query_1_5b](assets/screenshots/query1_5b.png)

## Section 2 - Elastic Agent Skills with Claude Code

### Section 2A — Install Skills

In [ ]:
%%bash
echo "--- Adding Elastic Agent Skills ---"
npx skills add elastic/agent-skills --claude < /dev/null > /dev/null 2>&1 && echo "Done."

echo "--- Installed Claude Skills ---"
ls .claude/skills/ | grep -E '^(elasticsearch|kibana|observability|security|cloud)-'


### Section 2B — Claude Code Queries

#### Filtered time-series — delivered orders over 90 days with BUCKET and date math

*"Show me daily order volume for the last 90 days, only for delivered orders"*

In [ ]:
%%bash
set -a && source .env && set +a
claude -p "Show me daily order volume for the last 90 days, only for delivered orders" \
"Show the Elastic agent skills being used, and the final answer. " \
--allowedTools Bash \
--model sonnet

#### Top-N + iterative refinement — top customers by spend, then a follow-up adding region to the BY clause in one prompt slot

*"Who are the top 10 customers by lifetime spend, and how many orders has each placed?"* 

In [ ]:
%%bash
set -a && source .env && set +a
claude -p "Who are the top 10 customers by lifetime spend, and how many orders has each placed?" \
"Show the Elastic agent skills being used, and the final answer. " \
--allowedTools Bash \
--model opus

#### Semantic search — fitness/recovery products spanning Sports, Electronics, and Clothing via product_description

*"Which products are customers buying for physical fitness and recovery, regardless of which category they're listed under? Use semantic search."*

In [ ]:
%%bash
set -a && source .env && set +a
claude -p "Which products are customers buying for physical fitness and recovery, " \
"regardless of which category they're listed under? Use semantic search. " \
"Show the Elastic agent skills being used, and the final answer. " \
--allowedTools Bash \
--model sonnet

#### Multi-condition aggregation — return rate by category vs. overall average

*"What is the return rate for each product category, and which categories are above the overall return rate?"*

In [ ]:
%%bash
set -a && source .env && set +a
claude -p "What is the return rate for each product category, and which categories are above the overall return rate?" \
"Show the Elastic agent skills being used, and the final answer." \
--allowedTools Bash \
--model opus

#### Full-text search — keyword MATCH across product descriptions

*"Find all products whose descriptions mention 'wireless' or 'bluetooth', and show total units sold and revenue for each"*

In [ ]:
%%bash
set -a && source .env && set +a
claude -p "Find all products whose descriptions mention 'wireless' or 'bluetooth', and show total units sold and revenue for each. Use a full-text MATCH search on the product_description field." \
"Show the Elastic agent skills being used, and the final answer." \
--allowedTools Bash \
--model sonnet

## Section 3 - ECH Deprovisioning

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."